# J3 PM | Traitement de données visuelles et textuelles

**Objectifs d'apprentissage :**
- OCR : Extraire le texte d'un document scanné complexe ("J'accuse") et le corriger via le Prompt Engineering.
- Extraire des images depuis un réseau social (Bluesky).
- Utiliser un modèle de reconnaissance visuelle (Vision API) pour décrire et quantifier la présence de symboles politiques.


In [ ]:
!apt-get update -qq && apt-get install -y tesseract-ocr tesseract-ocr-fra > /dev/null
!pip install pytesseract requests openai pandas matplotlib seaborn pillow


In [ ]:
import os
import requests
import json
import pytesseract
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from io import BytesIO
from openai import OpenAI

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

try:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")
except Exception as e:
    print("Erreur d'initialisation :", e)


## 1. OCR et correction par LLM : Lire les archives

Souvent en recherche (histoire politique, sociologie), on fait face à des scans de journaux anciens. Tesseract est un outil classique d'OCR, mais il fait des erreurs de mise en page.
Nous allons télécharger la célèbre une de "J'accuse" et utiliser un LLM (Prompt System) pour corriger et formater l'OCR brut.


In [ ]:

headers = {'User-Agent': 'Mozilla/5.0'}

# Téléchargement de la une de L'Aurore (J'accuse)
jaccuse_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/0/04/J%27accuse_-_Gallica_-_page_1.jpg/1280px-J%27accuse_-_Gallica_-_page_1.jpg"
response = requests.get(jaccuse_url, headers=headers)
jaccuse_img = Image.open(BytesIO(response.content))

# OCR brut
raw_text = pytesseract.image_to_string(jaccuse_img, lang='fra')

print("--- Extrait de l'OCR brut ---")
print(raw_text[:400])



Utilisons notre expertise en **Prompt Engineering** pour nettoyer ce texte.


In [ ]:

system_prompt = "Tu es un historien expert en traitement d'archives. Ton rôle est de corriger les coquilles d'un OCR brut et de structurer le texte lisiblement."

user_prompt = f"""
Voici le résultat brut de l'OCR de la une d'un journal historique ('J'accuse' de Zola).
1. Corrige les erreurs de l'OCR.
2. Ne garde que le début de la lettre (les 3 premiers paragraphes).

Texte OCR brut :
{raw_text[:1500]}
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.1
)
print("--- Texte corrigé par le LLM ---")
print(response.choices[0].message.content)



## 2. Analyse systématique d'images (Bluesky)

Nous allons récupérer les dernières images d'un compte politique sur Bluesky pour analyser la scénographie visuelle (présence de symboles).


In [ ]:

def get_bluesky_images(handle, max_images=5):
    url = f"https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed?actor={handle}"
    response = requests.get(url)
    images = []
    
    if response.status_code == 200:
        data = response.json()
        for item in data.get('feed', []):
            embed = item.get('post', {}).get('embed', {})
            
            img_list = []
            if embed.get('$type') == 'app.bsky.embed.images#view':
                img_list = embed.get('images', [])
            elif embed.get('$type') == 'app.bsky.embed.recordWithMedia#view':
                media = embed.get('media', {})
                if media.get('$type') == 'app.bsky.embed.images#view':
                    img_list = media.get('images', [])
                    
            for img in img_list:
                images.append({
                    'author': handle,
                    'image_url': img.get('thumb')
                })
                if len(images) >= max_images:
                    return images
    return images

# Récupérons les images du Parti Socialiste et d'un compte test riche en images
actor_1 = "partisocialiste.bsky.social"
actor_2 = "jay.bsky.team"

image_df = pd.DataFrame(get_bluesky_images(actor_1, 5) + get_bluesky_images(actor_2, 5))

# Visualisation rapide
if not image_df.empty:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i, row in image_df.iterrows():
        ax = axes[i // 5, i % 5]
        try:
            ax.imshow(Image.open(BytesIO(requests.get(row['image_url']).content)))
        except:
            pass
        ax.axis('off')
        ax.set_title(row['author'].split('.')[0])
    plt.tight_layout()
    plt.show()



## 3. Extraction de probabilités avec l'API Vision

Estimons la probabilité de présence de `drapeau`, `foule` et `costume` via un JSON structuré.


In [ ]:

def extract_symbols(image_url):
        
    prompt = '''
    Analyse cette image. Estime la probabilité (de 0 à 100) de présence de :
    - "drapeau": Drapeau
    - "foule": Foule ou public
    - "costume": Tenue formelle
    
    Réponds UNIQUEMENT avec un objet JSON au format :
    {"drapeau": 80, "foule": 0, "costume": 100}
    '''
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]}
            ],
            response_format={ "type": "json_object" },
            temperature=0,
        )
        return json.loads(response.choices[0].message.content)
    except:
        return {"drapeau": 0, "foule": 0, "costume": 0}

results = []
if not image_df.empty:
    for i, row in image_df.iterrows():
        symbols = extract_symbols(row['image_url'])
        symbols['author'] = row['author']
        results.append(symbols)

results_df = pd.DataFrame(results)
results_df.head()
